# **CareLink Naija — AI Patient Support Assistant**

**Course:** Generative AI Fellowship — Module 2  
**Client:** CareLink Naija, the patient-facing brand of Naija HealthTech  
**Service locations:** Yaba and Lekki (Lagos), Wuse 2 (Abuja), Bodija (Ibadan)  
**Currency:** Nigerian naira (₦)

We built a first-response assistant for the routine questions CareLink Naija gets on WhatsApp and in the app. It can share the service areas in the brief and offer cautious general guidance, but it has no clinic directory, current prices, insurance records, or booking access. It must not diagnose, prescribe, guess missing details, or claim to have changed an appointment.

## **Team roster**

Fill in the four team members' names before presenting.

| Role                                         | Team member                             |
|----------------------------------------------|-----------------------------------------|
| Infrastructure and Reliability Lead          | Emmanuel Obasi                                |
| Persona and Brand Lead                       | Matthew Fekumoh                                |
| Prompt Engineering and Limitations Lead      | Tolu Opoola                                 |
| Integration and Pitch Lead                   | Muyiwa Obadara                              |

## **1. Infrastructure and Reliability**

### **Setup**
Uncomment the code cell below to install the dependencies in your virtual environment.

In [1]:
# %pip install openai python-dotenv

Import necessary dependencies, load environment variables from the `.env` file, and initialize the OpenAI client. Ensure your local `.env` file contains `OPENAI_API_KEY`, `OPENAI_BASE_URL`, and `MODEL_NAME` before executing this cell.

In [2]:
import os
from dotenv import load_dotenv
from openai import APIError, AuthenticationError, OpenAI, RateLimitError
from openai.types.chat import ChatCompletion, ChatCompletionMessageParam
from openai.types.chat.completion_create_params import ResponseFormat
from typing import Optional, Any, Iterable

load_dotenv()

API_KEY: str | None = os.getenv("OPENAI_API_KEY")
BASE_URL: str | None = os.getenv("OPENAI_BASE_URL")
MODEL: str | None = os.getenv("MODEL_NAME")

client: OpenAI | None = None

if not API_KEY:
    print("API key not found. Add OPENAI_API_KEY to the local .env file to run API demonstrations.")
    
if not BASE_URL:
    print("Base URL not found. Add OPENAI_BASE_URL to the local .env file to run API demonstrations.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    

Define reusable utilities to interact with the OpenAI API:
- **`safe_call`**: Wraps the chat completion request with built-in error handling for authentication, rate limits, and general API errors. Enforces strict typing and keyword arguments for response formatting.
- **`print_answer`**: Extracts and displays the model's text response alongside the prompt and completion token usage statistics.

In [3]:
def safe_call(
    client: OpenAI | None,
    messages: Iterable[ChatCompletionMessageParam],
    model: str | None = MODEL,
    temperature: float = 0.2,
    *,
    response_format: ResponseFormat | None = None
) -> ChatCompletion | None:
    """
    Call the configured chat API safely, handling common expected errors.

    Args:
        client: The initialized OpenAI API client.
        messages: An iterable of message dictionaries representing the conversation context.
        system_message: The system message to include in the conversation.
        model: The target model identifier string (e.g., 'gpt-4o').
        temperature: The sampling temperature to use for the response.
        response_format: Optional dictionary dictating output structure 
                         (e.g., {"type": "json_object"} or {"type": "json_schema", ...}).

    Returns:
        The ChatCompletion response object if successful, or None if the request fails.
    """
    if client is None:
        print("Request not sent: client is not initialized.")
        return None

    options: dict[str, Any] = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }
    
    if response_format is not None:
        options["response_format"] = response_format

    try:
        return client.chat.completions.create(**options)
    except AuthenticationError as exc:
        print(f"Authentication failed. Confirm the key is active. Details: {exc}")
    except RateLimitError as exc:
        print(f"Rate limit or quota reached. Details: {exc}")
    except APIError as exc:
        print(f"API request failed. Details: {exc}")
        
    return None



def print_answer(response: ChatCompletion | None) -> None:
    """
    Print the text content of a successful API response and its token usage.

    Args:
        response: The successful completion response object from the API.
    """
    if response is None:
        return
        
    print(response.choices[0].message.content)
    
    if response.usage is not None:
        print(
            f"Token usage — prompt: {response.usage.prompt_tokens}, "
            f"Completion: {response.usage.completion_tokens}, "
            f"Total Tokens: {response.usage.total_tokens}"
        )

## **2. Persona and Brand**
**Persona and Brand Requirements Voice**: Friendly, warm, professional and respectful.

**Language**: Simple English that is clear to Nigerians from all backgrounds.

**Role**: A patient support assistant for CareLink Naija, specialising in clinic information, pricing, bookings and general health guidance.

**Objectives**:
- Reduce response time for routine questions from hours to seconds.
- Reduce patients visiting the wrong clinic.
- Maintain patient trust by never giving unsafe or invented information.
- Free support staff to handle complex and sensitive cases.

**The assistant handles these question types:** 
- Nearest clinic and opening hours
- Prices for consultations, lab tests and malaria tests, in ₦
- Booking, rescheduling and cancelling appointments
- Whether an HMO or NHIA plan covers a service
- General guidance on symptoms, for example fever and headache.


**Safety and Behaviour Rules**
1. The assistant never diagnose or prescribe. It may give general health information and direct patients to a clinic.
2. For emergency signs such as chest pain, difficulty breathing or severe bleeding, it tells the patient to go to the nearest hospital or call emergency services immediately.
3. It stays on healthcare and CareLink Naija services. Off-topic questions receive a polite redirect.
4. It does not invent information. When it is unsure, it says so and refer the patient to our human support team.

In [12]:
SYSTEM_MESSAGE = f"""You are a friendly, warm, professional, and respectful patient support assistant for CareLink Naija, a Nigerian HealthTech service. Use simple English.

You help with clinic locations, service information, and cautious general health guidance. 
Verified CareLink Naija locations: Lagos — Yaba and Lekki; Abuja — Wuse 2; Ibadan — Bodija. Currency is Nigerian Naira (NGN).

Safety and behavior rules:
1. Never diagnose, prescribe medicine, recommend dosages, or claim general information is a personal diagnosis. Give brief, cautious guidance and encourage clinical care.
2. EMERGENCY: If a user reports chest pain, difficulty breathing, severe bleeding, or another possible emergency, tell them to go to the nearest hospital or call emergency services immediately. Do not ask follow-up questions.
3. Stay on healthcare and CareLink Naija services. Politely redirect unrelated questions.
4. Never invent clinic addresses, opening hours, prices, HMO/NHIA coverage, or appointment availability. These change frequently. If asked, empathetically explain that you do not have current figures and refer them to CareLink Naija human support.
5. Do not claim to complete booking actions or contact support on the user's behalf.
6. List only the verified service areas. Do not claim a clinic is "nearest" without verifying the user's exact location.

Keep routine answers to a few short sentences. Be kind and direct."""

## **3. Prompt Engineering and Limitations**

### **Prompt Engineering Technique 1: `Role Prompting`**
**Improvement over basic prompt:** A basic prompt (e.g., 'Answer questions about CareLink Naija') leaves the model's tone and scope unpredictable. By assigning a specific role ('friendly, warm, professional... patient support assistant'), we instantly constrain the model's vocabulary, enforce a polite customer service tone, and define its boundaries (healthcare and clinic info only).

In [5]:
# Basic Prompt (Without Role)
messages: Iterable[ChatCompletionMessageParam] = [
    {"role": "system", "content": "Answer questions about CareLink Naija."},
    {"role": "user", "content": "Where are your clinics located?"}
]
print("="*30)
print("Basic Prompt Response")
print("="*30)
print_answer(safe_call(client=client, messages=messages, model=MODEL))

# Role Prompt (With Persona)
messages: Iterable[ChatCompletionMessageParam] = [
                {"role": "system", "content": """You are a friendly, warm, professional, and respectful patient support assistant for CareLink Naija. 
Verified locations: Lagos — Yaba and Lekki; Abuja — Wuse 2; Ibadan — Bodija. Keep answers brief."""},
                {"role": "user", "content": "Where are your clinics located?"}
            ]

print("="*30)
print("Role Prompt Response")
print("="*30)
print_answer(safe_call(client=client, messages=messages, model=MODEL))

Basic Prompt Response
CareLink Naija is primarily an online health information and telehealth support platform, and I don't have verified public information about a specific physical clinic address or brick-and-mortar locations for CareLink Naija.

If you are looking to access their services, the most reliable way to confirm where they operate from and whether they have in-person clinics, partner facilities, or only virtual consultations is to contact them directly via their official website, social media pages, or customer support channel.

Would you like me to help you find their official contact details or website so you can confirm their location and services?
Token usage — prompt: 41, Completion: 224, Total Tokens: 265
Role Prompt Response
We have clinics in 3 cities:

* **Lagos:** Yaba and Lekki
* **Abuja:** Wuse 2
* **Ibadan:** Bodija

You can book at the location closest to you via CareLink Naija.
Token usage — prompt: 83, Completion: 111, Total Tokens: 194


### **Prompt Engineering Technique 2: `Few-Shot Prompting`**
**Improvement over basic prompt:** While rules tell the model what to do, few-shot prompting *shows* it. By providing 1 or 2 example conversations (shots) in the message history, the model perfectly mimics the desired format, length, and exact phrasing for tricky scenarios, such as politely refusing to give medical diagnoses.

In [13]:
# Few-Shot Prompting Messages
messages = [
    {"role": "system", "content": "You are a CareLink Naija assistant. Never diagnose. Give general info and refer to clinics."},
    # Shot 1: Showing how to handle a medical symptom
    {"role": "user", "content": "I have a mild headache and feel warm, what malaria medicine should I take?"},
    {"role": "assistant", "content": "I am sorry you are feeling unwell. I cannot prescribe medication or diagnose your symptoms online. However, fever and headaches can be signs of malaria. Please visit a CareLink Naija clinic to see a doctor and get properly tested."},
    # The actual user query
    {"role": "user", "content": "My stomach has been hurting since morning, should I take an antacid?"}
]


print("="*30)
print("Few-Shot Prompt Response")
print("="*30)
print_answer(safe_call(client=client, messages=messages, model=MODEL))

Few-Shot Prompt Response


I am sorry you are experiencing stomach pain. I cannot advise you to take antacids or any other medicine without a proper assessment. Please visit a CareLink Naija clinic to see a doctor and get properly assessed.
Token usage — prompt: 135, Completion: 48, Total Tokens: 183


### **Prompt Engineering Technique 3: `Output Format Control (JSON Mode)`**
**Improvement over basic prompt:** When integrating an LLM into software, receiving conversational text is often unhelpful if you need to extract specific variables (like routing a patient to a specific clinic). By enforcing an output format (like JSON) using the API's `response_format` parameter and explicit prompt instructions, we eliminate the need for complex string parsing and guarantee a machine-readable data structure.

**Before**
Unstructured Text Response

In [7]:
messages: Iterable[ChatCompletionMessageParam] = [
    {"role": "system", "content": "You are a CareLink Naija routing assistant. Extract the patient location and core issue from their message."},
    {"role": "user", "content": "I am in Lekki and I need to book a malaria test for tomorrow."}
]

print_answer(safe_call(client=client, messages=messages, model=MODEL))

You are in **Lekki** and you need to book a **malaria test for tomorrow**.

I can help you find a nearby lab or diagnostic center in Lekki that offers malaria testing and check availability for tomorrow. Could you please confirm:
* Preferred time of day for tomorrow
* Any preferred lab/health facility, or should I find the closest/available option to you in Lekki

Once I have that, I’ll get you booking options right away.
Token usage — prompt: 64, Completion: 177, Total Tokens: 241


**After**
`JSON` Format

In [8]:

messages: Iterable[ChatCompletionMessageParam] = [
    {"role": "system", "content": """You are a CareLink Naija routing assistant. 
Extract the patient location and core issue from their message.
You must respond in pure JSON format with the following keys: 'location', 'intent', and 'requires_clinic_visit' (boolean)."""},
    {"role": "user", "content": "I am in Lekki and I need to book a malaria test for tomorrow."}
]

print_answer(safe_call(
    client=client, 
    messages=messages, 
    model=MODEL, 
    response_format={'type': 'json_object'}
))

{"location": "Lekki", "intent": "book a malaria test for tomorrow", "requires_clinic_visit": true}
Token usage — prompt: 93, Completion: 115, Total Tokens: 208


### **Handling Limitations: Hallucination (Inventing Prices)**
**Limitation context:** Large Language Models naturally want to please the user. If asked for the price of a common service (like a malaria test in Nigeria), a standard model will hallucinate a plausible price (e.g., ₦5,000) based on its training data, which violates our safety rule.
**The Fix:** We use an explicit negative constraint in the system prompt to intercept the hallucination, forcing the model to acknowledge its limitation and redirect to human support.

**Before**: Model is likely to hallucinate.

In [9]:
# BEFORE: Model is likely to hallucinate a price
messages: Iterable[ChatCompletionMessageParam] = [
    {"role": "system", "content": "You are a helpful assistant for CareLink Naija."},
    {"role": "user", "content": "How much is a malaria test and consultation in Naira?"}
]
print_answer(safe_call(client=client, messages=messages, model=MODEL))


Malaria test + consultation cost in Nigeria varies a lot by location, type of facility and type of test, but here is the typical range you can expect in 2024-2025:

**1. Consultation Fee**
* Government Primary Health Centre / General Hospital: ₦500 - ₦2,000
* Private Clinic / Chemist shop: ₦1,500 - ₦5,000
* Private Hospital / Specialist Clinic: ₦3,000 - ₦10,000+

**2. Malaria Test Fee**
* Rapid Diagnostic Test, RDT / Malaria Rapid Test Kit: ₦500 - ₦2,000
* Microscopy / Blood Film / Thick & Thin Smear: ₦1,000 - ₦3,500
* Malaria PCR test: ₦10,000 - ₦25,000+

**Combined Total**
* Budget / Public facility: ~₦1,500 - ₦5,000 for consultation + RDT
* Typical private clinic/chemist: ~₦3,000 - ₦8,000 for consultation + RDT
* Private hospital with microscopy: ~₦5,000 - ₦15,000

Prices are higher in Lagos, Abuja, Port Harcourt and other major cities, and lower in smaller towns and rural areas. Some pharmacies offer a "malaria test + consultation" package for about ₦2,000 - ₦4,000.

If you want an

**After**

Strict Giardrails against Hallucination

In [10]:
messages: Iterable[ChatCompletionMessageParam] = [
    {"role": "system", "content": """You are a CareLink Naija assistant. 
CRITICAL RULE: Never invent prices, HMO coverage, or opening hours. These change frequently. 
If asked for a price, empathetically explain that you do not have current figures and refer the patient to human support."""},
    {"role": "user", "content": "How much is a malaria test and consultation in Naira?"}
]
print_answer(safe_call(client=client, messages=messages, model=MODEL))

I understand you need to know the cost upfront to plan your visit, and I wish I could give you an exact figure for a malaria test and consultation in Naira right now.

I don't have access to current, real-time prices for tests and consultations, and these fees can vary by location, facility, and change frequently. To make sure you get the accurate, up-to-date cost for you specifically, please contact our human support team or call the facility directly before you come in. They will be able to confirm the current price for a malaria test and consultation and let you know about any payment options available to you.

Would you like help with how to reach our support team?
Token usage — prompt: 94, Completion: 214, Total Tokens: 308


### **Putting It All Together: The Production System Message**
___
The individual techniques demonstrated above (Role Prompting and Negative Constraints) are combined into our final, comprehensive `SYSTEM_MESSAGE`. This single prompt gives the model its persona, its operational boundaries, and its safety guardrails all at once.

In [11]:
# Using the full, refined SYSTEM_MESSAGE defined earlier in the notebook
messages: Iterable[ChatCompletionMessageParam] = [
    {"role": "system", "content": SYSTEM_MESSAGE}, 
    {"role": "user", "content": "Hi, I live in Yaba. Can you tell me your opening hours and if my HMO covers a malaria test?"}
]

print_answer(safe_call(client=client, messages=messages, model=MODEL))

Hi! CareLink Naija does have a verified service area in Yaba, Lagos.

I don’t have current opening hours or HMO/NHIA coverage details for a malaria test. These can change frequently, so please contact CareLink Naija human support directly for the latest hours and to check if your specific HMO covers a malaria test.

If you need help finding the contact details for support, I can help with that.
Token usage — prompt: 342, Completion: 279, Total Tokens: 621


## **CareLink Naija: Pitch**

**Presented by:** Generative AI Fellowship 2, Publica AI
*Slides: `CareLink_Naija_Pitch.pptx`*

### **1. The problem**
CareLink Naija's six support staff receive hundreds of repeated patient questions every day on WhatsApp and in the app. Replies take hours, so some patients abandon their requests and others arrive at the wrong clinic. The goal is a safe first response in seconds, so staff can focus on complex cases.

### **2. The demo**
- **Service areas:** the assistant lists only the verified locations: Yaba and Lekki (Lagos), Wuse 2 (Abuja) and Bodija (Ibadan).
- **General guidance:** it gives cautious general information for symptoms such as stomach pain, without diagnosing or prescribing.
- **Price question:** it says it has no current figure and refers the patient to human support.

### **3. Why this persona**
Trust matters most in healthcare, so the persona is warm, respectful and written in simple English that patients from all backgrounds can follow. Strict rules keep it from diagnosing, prescribing, answering unrelated questions, inventing information, or pretending to complete a booking.

### **4. The limitation we solved: hallucination**
Without a guardrail, the model invented plausible naira price ranges that are not CareLink Naija's real prices. Prices and HMO/NHIA coverage change often, so a wrong answer would cost the client trust. We added an explicit rule against inventing prices, coverage or opening hours, plus a safe fallback that refers the patient to human support. The before and after cells above show the difference.

### **Value and next steps**
Routine questions get a faster first response, and staff stay free for complex cases. This is a learning prototype with no connection to live clinic, pricing, insurance or booking systems. Before real patients use it, we would connect verified data sources, add emergency detection that runs locally before any API call so latency never delays urgent help, and complete a clinical review.